# 🍷 Classificação da Qualidade de Vinhos com Machine Learning
## POSTECH — Tech Challenge Fase 2

**Dataset:** Wine Quality Dataset (WineQT.csv)  
**Objetivo:** Classificar vinhos como *Alta Qualidade* (nota ≥ 7) ou *Baixa/Média Qualidade* (nota < 7) utilizando características físico-químicas.

---
### Pipeline
1. Compreensão do Problema
2. Análise Exploratória de Dados (EDA)
3. Pré-processamento
4. Desenvolvimento dos Modelos
5. Avaliação e Comparação
6. Interpretação dos Resultados

In [1]:
# Importações
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report
from preprocessing import load_data, create_binary_target, feature_engineering, preprocess
from modeling import (
    train_models, compare_models,
    plot_confusion_matrices, plot_roc_curves,
    plot_feature_importance, plot_metrics_comparison
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42
print('Bibliotecas importadas com sucesso ✓')

Bibliotecas importadas com sucesso ✓


---
## 1. Compreensão do Problema

A avaliação da qualidade de vinhos é, tradicionalmente, um processo subjetivo realizado por sommeliers e enólogos. Este projeto propõe um modelo de Machine Learning capaz de **prever automaticamente** se um vinho é de alta qualidade com base em dados físico-químicos mensuráveis.

### Variável Alvo (Target)

A coluna `quality` (nota de 0 a 10 atribuída por especialistas) é transformada em classificação binária:

| Classe | Critério | Rótulo |
|--------|----------|--------|
| 1 | quality ≥ 7 | Alta Qualidade |
| 0 | quality < 7 | Baixa/Média Qualidade |

In [2]:
# Carregamento dos dados
df = load_data('../data/WineQT.csv')

print(f'Shape: {df.shape}')
print(f'Variáveis: {list(df.columns)}')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head()

Shape: (1143, 12)
Variáveis: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
Valores nulos: 0


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
# Estatísticas descritivas
df.describe().round(3)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000,1143.000
mean,8.311,0.531,0.268,2.532,0.087,15.615,45.915,0.997,3.311,0.658,10.442,5.657
std,1.748,0.180,0.197,1.356,0.047,10.250,32.782,0.002,0.157,0.170,1.082,0.806
min,4.600,0.120,0.000,0.900,0.012,1.000,6.000,0.990,2.740,0.330,8.400,3.000
25%,7.100,0.392,0.090,1.900,0.070,7.000,21.000,0.996,3.205,0.550,9.500,5.000
50%,7.900,0.520,0.250,2.200,0.079,13.000,37.000,0.997,3.310,0.620,10.200,6.000
75%,9.100,0.640,0.420,2.600,0.090,21.000,61.000,0.998,3.400,0.730,11.100,6.000
max,15.900,1.580,1.000,15.500,0.611,68.000,289.000,1.004,4.010,2.000,14.900,8.000


In [4]:
# Criação do target binário
df_bin = create_binary_target(df.copy())
class_counts = df_bin['high_quality'].value_counts()
print('Distribuição do target binário:')
print(f'  Alta Qualidade (1):    {class_counts[1]} amostras ({class_counts[1]/len(df_bin)*100:.1f}%)')
print(f'  Baixa/Média (0):       {class_counts[0]} amostras ({class_counts[0]/len(df_bin)*100:.1f}%)')

Distribuição do target binário:
  Alta Qualidade (1):    159 amostras (13.9%)
  Baixa/Média (0):       984 amostras (86.1%)


---
## 2.1 Qualidade dos Dados — Auditoria Completa

Antes de qualquer análise, auditamos o dataset em 5 dimensões:
**valores nulos**, **duplicatas**, **tipos de dados**, **valores impossíveis** e **zeros suspeitos**.
Cada decisão de limpeza é justificada abaixo.

In [5]:
import pandas as pd
import numpy as np

df_raw_audit = pd.read_csv('../data/WineQT.csv')

print('=' * 55)
print('AUDITORIA DE QUALIDADE DOS DADOS')
print('=' * 55)

# 1. Shape
print(f'\n📐 Dimensões originais: {df_raw_audit.shape[0]} linhas × {df_raw_audit.shape[1]} colunas')

# 2. Nulos
nulls = df_raw_audit.isnull().sum()
print(f'\n🔍 Valores nulos por coluna:')
if nulls.sum() == 0:
    print('   ✅ Nenhum valor nulo encontrado — dataset completo.')
else:
    print(nulls[nulls > 0])

# 3. Duplicatas
dup_rows = df_raw_audit.duplicated().sum()
dup_ids  = df_raw_audit['Id'].duplicated().sum()
print(f'\n🔁 Duplicatas:')
print(f'   Linhas completamente duplicadas : {dup_rows}')
print(f'   IDs duplicados                  : {dup_ids}')
if dup_rows == 0:
    print('   ✅ Nenhuma duplicata encontrada.')

# 4. Tipos de dados
print(f'\n📊 Tipos de dados:')
print(df_raw_audit.dtypes.to_string())
print('   ✅ Todos os tipos estão corretos (float64 para variáveis contínuas, int64 para quality/Id).')

# 5. Valores negativos (impossíveis fisicamente)
feat_cols = [c for c in df_raw_audit.columns if c not in ['Id', 'quality']]
negs = {col: (df_raw_audit[col] < 0).sum() for col in feat_cols if (df_raw_audit[col] < 0).sum() > 0}
print(f'\n⚠️  Valores negativos (fisicamente impossíveis):')
if not negs:
    print('   ✅ Nenhum valor negativo — todos os valores respeitam os limites físicos.')
else:
    for k, v in negs.items():
        print(f'   {k}: {v} ocorrências')

# 6. Zeros
print(f'\n🔢 Zeros por feature (podem indicar dado faltante disfarçado):')
zeros = {col: (df_raw_audit[col] == 0).sum() for col in feat_cols if (df_raw_audit[col] == 0).sum() > 0}
for col, cnt in zeros.items():
    print(f'   {col}: {cnt} zeros ({cnt/len(df_raw_audit)*100:.1f}%)')

# 7. Coluna Id
print(f'\n🪪  Coluna Id:')
print(f'   Vai de {df_raw_audit["Id"].min()} a {df_raw_audit["Id"].max()} (esperado 0–1143)')
print(f'   Valores únicos: {df_raw_audit["Id"].nunique()} — indica que o dataset original')
print(f'   tinha ~1598 registros e este é um subconjunto amostrado.')

print('\n' + '=' * 55)
print('RESUMO DAS DECISÕES DE LIMPEZA')
print('=' * 55)

AUDITORIA DE QUALIDADE DOS DADOS

📐 Dimensões originais: 1143 linhas × 13 colunas

🔍 Valores nulos por coluna:
   ✅ Nenhum valor nulo encontrado — dataset completo.

🔁 Duplicatas:
   Linhas completamente duplicadas : 0
   IDs duplicados                  : 0
   ✅ Nenhuma duplicata encontrada.

📊 Tipos de dados:
fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
Id                        int64
   ✅ Todos os tipos estão corretos (float64 para variáveis contínuas, int64 para quality/Id).

⚠️  Valores negativos (fisicamente impossíveis):
   ✅ Nenhum valor negativo — todos os valores respeitam os limites físicos.

🔢 Zeros por feature (podem indicar dado faltante di

### 🧹 Decisões de Limpeza — Justificativas

| # | Ponto Analisado | Resultado | Decisão | Justificativa |
|---|---|---|---|---|
| 1 | **Valores nulos** | 0 nulos | ✅ Nenhuma ação | Dataset completo, sem necessidade de imputação |
| 2 | **Linhas duplicadas** | 0 duplicatas | ✅ Nenhuma ação | Todas as amostras são únicas |
| 3 | **Tipos de dados** | Corretos | ✅ Nenhuma ação | float64 e int64 adequados para as variáveis |
| 4 | **Valores negativos** | Nenhum | ✅ Nenhuma ação | Todos os valores são fisicamente possíveis |
| 5 | **Zeros em `citric acid`** | 99 zeros (8.7%) | ✅ **Mantidos** | Zeros são **tecnicamente válidos**: muitos vinhos tintos não recebem adição de ácido cítrico durante a produção. Não representa dado faltante |
| 6 | **Coluna `Id`** | IDs de 0 a 1597 (não sequencial) | 🗑️ **Removida** | Identificador de linha sem valor preditivo. Sua escala poderia introduzir ruído nos modelos |

> **Conclusão:** O dataset chegou em excelente estado. A única intervenção necessária foi **remover a coluna `Id`**, que não carrega informação sobre a qualidade do vinho.

---
## 2. Análise Exploratória de Dados (EDA)

In [6]:
# Distribuição da qualidade original
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma com notas originais
quality_counts = df['quality'].value_counts().sort_index()
colors = ['#e63946' if v < 7 else '#2a9d8f' for v in quality_counts.index]
axes[0].bar(quality_counts.index, quality_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribuição das Notas de Qualidade', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nota de Qualidade')
axes[0].set_ylabel('Contagem')
for i, (idx, val) in enumerate(quality_counts.items()):
    axes[0].text(idx, val + 3, str(val), ha='center', fontsize=9)
axes[0].grid(axis='y', alpha=0.4)

from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='#e63946', label='Baixa/Média (< 7)'),
    Patch(facecolor='#2a9d8f', label='Alta (≥ 7)')
])

# Pie chart do target binário
axes[1].pie(class_counts.values, labels=['Baixa/Média\n(< 7)', 'Alta\n(≥ 7)'],
            colors=['#e63946', '#2a9d8f'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Balanceamento Binário das Classes', fontsize=12, fontweight='bold')

plt.suptitle('Variável Alvo: Qualidade do Vinho', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/01_quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n⚠️  Classes desbalanceadas: 86.1% baixa/média vs 13.9% alta qualidade')
print('   Estratégia: class_weight="balanced" nos modelos + métricas F1/AUC como foco principal')


⚠️  Classes desbalanceadas: 86.1% baixa/média vs 13.9% alta qualidade
   Estratégia: class_weight="balanced" nos modelos + métricas F1/AUC como foco principal


C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\444745278.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Histogramas de todas as features
features = [c for c in df.columns if c != 'quality']
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=30, color='#457b9d', edgecolor='white', alpha=0.85)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frequência')
    axes[i].grid(alpha=0.3)

for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribuição das Variáveis Físico-Químicas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/03_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\4181165030.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Matriz de correlação
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Matriz de Correlação — Variáveis Físico-Químicas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/04_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlação com quality
print('\nCorrelação com a variável quality (ordenada):')
print(corr['quality'].drop('quality').sort_values(ascending=False).round(3))


Correlação com a variável quality (ordenada):
alcohol                 0.485
sulphates               0.258
citric acid             0.241
fixed acidity           0.122
residual sugar          0.022
pH                     -0.052
free sulfur dioxide    -0.063
chlorides              -0.124
density                -0.175
total sulfur dioxide   -0.183
volatile acidity       -0.407
Name: quality, dtype: float64


C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\744271086.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Insights da Correlação

| Feature | Correlação com Quality | Interpretação |
|---|---|---|
| **alcohol** | +0.48 | Vinhos com maior teor alcoólico tendem a ter nota mais alta |
| **volatile acidity** | -0.39 | Alta acidez volátil (vinagre) prejudica a qualidade |
| **sulphates** | +0.25 | Aditivo que atua como antioxidante, melhora a conservação |
| **citric acid** | +0.23 | Confere frescor e sabor ao vinho |
| **density** | -0.17 | Vinhos mais densos (mais açúcar) tendem a ter nota menor |
| **total sulfur dioxide** | -0.19 | Excesso de SO₂ indica problemas de conservação |

In [9]:
# Boxplots: features vs qualidade binária
df_eda = df.copy()
df_eda['Qualidade'] = df_eda['quality'].apply(lambda x: 'Alta (≥7)' if x >= 7 else 'Baixa/Média (<7)')

key_features = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid',
                'density', 'total sulfur dioxide']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sns.boxplot(data=df_eda, x='Qualidade', y=feat,
                hue='Qualidade',
                palette={'Alta (≥7)': '#2a9d8f', 'Baixa/Média (<7)': '#e63946'},
                ax=axes[i], legend=False)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle('Distribuição das Principais Features por Qualidade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/05_features_vs_quality.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\1405589721.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Detecção de outliers (IQR)
print('Outliers detectados por feature (método IQR):')
outlier_report = []
for feat in features:
    Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[feat] < Q1 - 1.5*IQR) | (df[feat] > Q3 + 1.5*IQR)).sum()
    outlier_report.append({'Feature': feat, 'Outliers': n_out, '%': f'{n_out/len(df)*100:.1f}%'})
    print(f'  {feat:30s}: {n_out:3d} ({n_out/len(df)*100:.1f}%)')

print('\n⚠️  Outliers mantidos: representam variações naturais do processo produtivo.')
print('   Remoção poderia eliminar amostras de qualidade extrema (notas 3 e 8).')

Outliers detectados por feature (método IQR):
  fixed acidity                 :  44 (3.8%)
  volatile acidity              :  14 (1.2%)
  citric acid                   :   1 (0.1%)
  residual sugar                : 110 (9.6%)
  chlorides                     :  77 (6.7%)
  free sulfur dioxide           :  18 (1.6%)
  total sulfur dioxide          :  40 (3.5%)
  density                       :  36 (3.1%)
  pH                            :  20 (1.7%)
  sulphates                     :  43 (3.8%)
  alcohol                       :  12 (1.0%)

⚠️  Outliers mantidos: representam variações naturais do processo produtivo.
   Remoção poderia eliminar amostras de qualidade extrema (notas 3 e 8).


---
## 3. Pré-processamento de Dados

In [11]:
# Verificação de dados faltantes
print('Dados faltantes por coluna:')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '  Nenhum dado faltante encontrado ✓')

# Feature Engineering
print('\nFeatures criadas via Feature Engineering:')
print('  acidity_ratio         : fixed acidity / volatile acidity (equilíbrio ácido)')
print('  sulfur_ratio          : free SO₂ / total SO₂ (eficiência do conservante)')
print('  alcohol_density_ratio : alcohol / density (leveza alcoólica do vinho)')

Dados faltantes por coluna:
  Nenhum dado faltante encontrado ✓

Features criadas via Feature Engineering:
  acidity_ratio         : fixed acidity / volatile acidity (equilíbrio ácido)
  sulfur_ratio          : free SO₂ / total SO₂ (eficiência do conservante)
  alcohol_density_ratio : alcohol / density (leveza alcoólica do vinho)


In [12]:
# Pré-processamento completo
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess(df)

print(f'Conjunto de treino: {X_train.shape}')
print(f'Conjunto de teste:  {X_test.shape}')
print(f'Features totais: {len(feature_names)}')
print(f'\nFeatures: {feature_names}')
print(f'\nDistribuição no treino — Alta: {y_train.sum()} | Baixa/Média: {(y_train==0).sum()}')
print(f'Distribuição no teste  — Alta: {y_test.sum()} | Baixa/Média: {(y_test==0).sum()}')

Conjunto de treino: (914, 14)
Conjunto de teste:  (229, 14)
Features totais: 14

Features: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'acidity_ratio', 'sulfur_ratio', 'alcohol_density_ratio']

Distribuição no treino — Alta: 127 | Baixa/Média: 787
Distribuição no teste  — Alta: 32 | Baixa/Média: 197


---
## 4. Desenvolvimento dos Modelos

Foram treinados **três modelos** de classificação:

| Modelo | Abordagem | Pontos Fortes |
|---|---|---|
| **Regressão Logística** | Linear | Interpretável, baseline robusto |
| **Random Forest** | Ensemble (Bagging) | Robusto a outliers, captura não-linearidades |
| **Gradient Boosting** | Ensemble (Boosting) | Alta performance em dados tabulares |

In [13]:
# Treinamento dos modelos
models = train_models(X_train, y_train, random_state=SEED)
print('\nTodos os modelos treinados com sucesso ✓')

  ✓ Logistic Regression treinado.
  ✓ Random Forest treinado.
  ✓ Gradient Boosting treinado.

Todos os modelos treinados com sucesso ✓


---
## 5. Avaliação dos Modelos

### Métricas utilizadas
- **Acurácia**: % de predições corretas (pode ser enganosa com classes desbalanceadas)
- **Precisão**: dos vinhos classificados como alta qualidade, quantos realmente são?
- **Recall**: dos vinhos que são realmente de alta qualidade, quantos capturamos?
- **F1-Score**: média harmônica entre Precisão e Recall (métrica principal deste problema)
- **ROC-AUC**: capacidade discriminativa do modelo (independente do threshold)

In [14]:
# Comparação de métricas
results_df = compare_models(models, X_test, y_test)
print('Métricas de Avaliação:\n')
display(results_df.round(4).style
        .highlight_max(color='#c8f7c5', subset=['accuracy','precision','recall','f1_score','roc_auc'])
        .format('{:.4f}'))

results_df.round(4).to_csv('../results/metrics_comparison.csv')
print('\n✓ Métricas exportadas para results/metrics_comparison.csv')

Métricas de Avaliação:



,accuracy,precision,recall,f1_score,roc_auc
model,,,,,
Logistic Regression,0.7904,0.3621,0.6562,0.4667,0.8637
Random Forest,0.8996,0.6667,0.5625,0.6102,0.9036
Gradient Boosting,0.9127,0.7000,0.6562,0.6774,0.8831



✓ Métricas exportadas para results/metrics_comparison.csv


In [15]:
# Gráfico comparativo de métricas
plot_metrics_comparison(results_df, save_path='../results/08_metrics_comparison.png')

img = plt.imread('../results/08_metrics_comparison.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\4073878501.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Matrizes de confusão
plot_confusion_matrices(models, X_test, y_test, save_path='../results/06_confusion_matrices.png')

img = plt.imread('../results/06_confusion_matrices.png')
plt.figure(figsize=(18, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\1734534231.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Curvas ROC
plot_roc_curves(models, X_test, y_test, save_path='../results/07_roc_curves.png')

img = plt.imread('../results/07_roc_curves.png')
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\1937780163.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Classification report detalhado do melhor modelo
best_model_name = results_df['f1_score'].idxmax()
best_model = models[best_model_name]
y_pred = best_model.predict(X_test)

print(f'📊 Relatório Completo — {best_model_name}\n')
print(classification_report(y_test, y_pred,
                            target_names=['Baixa/Média Qualidade', 'Alta Qualidade']))

📊 Relatório Completo — Gradient Boosting

                       precision    recall  f1-score   support

Baixa/Média Qualidade       0.94      0.95      0.95       197
       Alta Qualidade       0.70      0.66      0.68        32

             accuracy                           0.91       229
            macro avg       0.82      0.81      0.81       229
         weighted avg       0.91      0.91      0.91       229



---
## 6. Interpretação dos Resultados

In [19]:
# Feature Importance — Random Forest
plot_feature_importance(models['Random Forest'], feature_names, 'Random Forest',
                        save_path='../results/09_feature_importance_Random_Forest.png')
img = plt.imread('../results/09_feature_importance_Random_Forest.png')
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Feature Importance — Random Forest', fontsize=13)
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\4002893024.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Feature Importance — Gradient Boosting
plot_feature_importance(models['Gradient Boosting'], feature_names, 'Gradient Boosting',
                        save_path='../results/09_feature_importance_Gradient_Boosting.png')
img = plt.imread('../results/09_feature_importance_Gradient_Boosting.png')
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Feature Importance — Gradient Boosting', fontsize=13)
plt.show()

C:\Users\KiKi\AppData\Local\Temp\ipykernel_20456\119709724.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Conclusões e Implicações para a Produção

### 🏆 Melhor Modelo: Gradient Boosting
- **F1-Score: 0.677** | **ROC-AUC: 0.883** | **Acurácia: 91.3%**

### 📌 Fatores que mais influenciam a qualidade

| Fator | Direção | Ação Recomendada |
|---|---|---|
| **Teor Alcoólico** | ↑ melhora qualidade | Controlar nível de fermentação |
| **Acidez Volátil** | ↓ piora qualidade | Monitorar acidificação bacteriana |
| **Sulfatos** | ↑ melhora qualidade | Ajustar dosagem do aditivo |
| **Ácido Cítrico** | ↑ melhora qualidade | Manter frescor natural da uva |
| **Dióxido de Enxofre** | ↓ piora em excesso | Otimizar concentração de conservante |

### 💡 Implicações Práticas
1. **Monitoramento em tempo real**: Medir alcohol e volatile acidity durante a fermentação pode antecipar a qualidade final.
2. **Controle de qualidade automatizado**: O modelo pode ser integrado ao processo produtivo para alertar sobre lotes com risco de baixa qualidade.
3. **Redução de custos**: Diminui a dependência de avaliações sensoriais manuais, agilizando o processo de certificação.
4. **Dataset desbalanceado**: Apenas 13.9% dos vinhos são de alta qualidade — recomenda-se técnicas de oversampling (SMOTE) em trabalhos futuros para melhorar o Recall.